# Aula 6 — Transformers: Self-Attention e Encoder (PyTorch)

**CCM-109 — Deep Learning · Prof. Ronaldo Prati (UFABC)**

Este notebook implementa o mecanismo de Transformer passo a passo:

1. **Self-attention do zero** — produto interno, softmax, média ponderada
2. **Positional encoding** — embeddings sinusoidais
3. **Multi-head attention** — projeções Q, K, V
4. **Bloco Encoder completo** — MHA + Add&Norm + FFN
5. **Aplicação: NER/Slot Filling** — classificação token a token com CoNLL-2003
6. **Visualização de atenção** — o que cada cabeça aprende
7. **Comparação com BiLSTM**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/REPO/blob/main/notebooks/lec06-transformers-torch.ipynb)

## 0. Setup

In [ ]:
# !pip install datasets seaborn --quiet


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

torch.manual_seed(42)
np.random.seed(42)

## 1. Self-Attention do Zero

Antes de usar o `nn.MultiheadAttention` do PyTorch, vamos implementar a atenção manualmente para entender cada passo.

**Fórmula:**
$$\text{Attention}(W) = \text{softmax}\!\left(\frac{WW^\top}{\sqrt{d}}\right)W$$

Onde $W$ é a matriz de embeddings (N × d), cada linha é o embedding de uma palavra.

In [ ]:
def self_attention_manual(W):
    """
    Self-attention simplificada (sem projeções Q/K/V).
    W: tensor (N, d) — embeddings das N palavras
    Retorna: (N, d) representações contextuais
    """
    N, d = W.shape

    # 1. Scores: todos os N×N produtos internos
    scores = W @ W.T                      # (N, N)

    # 2. Escala para estabilidade numérica
    scores = scores / math.sqrt(d)        # (N, N)

    # 3. Softmax linha a linha → pesos de atenção
    attn = F.softmax(scores, dim=-1)      # (N, N)

    # 4. Média ponderada dos embeddings
    out = attn @ W                        # (N, d)

    return out, attn


# Exemplo da aula: train station nearby the radio station (d=2)
words = ['train', 'sta₁', 'radio', 'sta₂']
W = torch.tensor([
    [0.9, 0.1],   # train
    [0.6, 0.6],   # station₁
    [0.1, 0.9],   # radio
    [0.6, 0.6],   # station₂ (embedding idêntico a station₁!)
], dtype=torch.float32)

out, attn = self_attention_manual(W)

print("Embeddings originais W (N×2):")
for w, e in zip(words, W):
    print(f"  {w:6s}: {e.tolist()}")

print("\nMatriz de atenção (softmax(WW^T/sqrt(d))):")
print(f"{'':7s}" + "".join(f"{w:7s}" for w in words))
for i, w in enumerate(words):
    row = [f"{attn[i,j]:.3f}" for j in range(len(words))]
    print(f"  {w:5s}: " + "  ".join(row))

print("\nRepresentações contextuais:")
for w, e in zip(words, out):
    print(f"  {w:6s}: {[round(x,3) for x in e.tolist()]}")

print("\n⚠️  sta₁ e sta₂ produzem representações idênticas — motivação para Positional Encoding!")

In [ ]:
# Visualizar a matriz de atenção
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(attn.numpy(), annot=True, fmt='.3f', cmap='Blues',
            xticklabels=words, yticklabels=words, ax=axes[0])
axes[0].set_title('Pesos de atenção\n(sem projeções Q/K/V)', fontsize=12)
axes[0].set_xlabel('atende a →')
axes[0].set_ylabel('palavra alvo')

# Visualizar embeddings originais vs. contextuais
colors = ['#3b82f6', '#eab308', '#a855f7', '#eab308']
for i, (w, c) in enumerate(zip(words, colors)):
    axes[1].annotate('', xy=out[i].tolist(), xytext=[0, 0],
                     arrowprops=dict(arrowstyle='->', color=c, lw=2))
    axes[1].annotate('', xy=W[i].tolist(), xytext=[0, 0],
                     arrowprops=dict(arrowstyle='->', color=c, lw=1, linestyle='--', alpha=0.4))
    axes[1].text(out[i][0].item()+0.01, out[i][1].item()+0.01, f'{w}\'')
    axes[1].text(W[i][0].item()+0.01, W[i][1].item()+0.01, w, alpha=0.4)

axes[1].set_xlim(-0.1, 1.1); axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title("Embeddings: originais (tracejado) vs. contextuais (sólido)")
axes[1].set_xlabel('d₁'); axes[1].set_ylabel('d₂')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Positional Encoding Sinusoidal

O Transformer original (Vaswani et al., 2017) usa encoding sinusoidal:

$$PE(pos, 2k) = \sin\!\left(\frac{pos}{10000^{2k/d}}\right), \quad PE(pos, 2k+1) = \cos\!\left(\frac{pos}{10000^{2k/d}}\right)$$

- `pos` = posição do token na sequência
- `k` = índice da dimensão do embedding
- `d` = dimensão total do embedding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Pré-computar a tabela de encodings (max_len × d_model)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()       # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )                                                          # (d_model/2,)

        pe[:, 0::2] = torch.sin(pos * div)   # dimensões pares
        pe[:, 1::2] = torch.cos(pos * div)   # dimensões ímpares

        # Registrar como buffer (não é parâmetro treinável)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# Visualizar os encodings
d_model = 64
max_len = 100
pe_layer = PositionalEncoding(d_model, max_len, dropout=0.0)
pe_matrix = pe_layer.pe[0].numpy()  # (max_len, d_model)

plt.figure(figsize=(14, 4))
plt.subplot(1, 2, 1)
plt.imshow(pe_matrix.T, aspect='auto', cmap='RdBu', origin='lower',
           extent=[0, max_len, 0, d_model])
plt.colorbar()
plt.xlabel('Posição na sequência')
plt.ylabel('Dimensão do embedding')
plt.title(f'Positional Encoding sinusoidal\n(d={d_model})')

plt.subplot(1, 2, 2)
for dim in [0, 1, 4, 5, 20, 21]:
    plt.plot(pe_matrix[:50, dim], label=f'd={dim}')
plt.xlabel('Posição')
plt.ylabel('Valor')
plt.title('Primeiras 50 posições — 6 dimensões selecionadas')
plt.legend(fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Cada posição tem um padrão único de sins/cossenos — o modelo aprende a distingui-las.")

## 3. Multi-Head Attention com Projeções Q, K, V

A atenção completa usa três projeções lineares independentes:
- **Q** (Query): o que a palavra está buscando?
- **K** (Key): o que a palavra tem para oferecer?
- **V** (Value): o que ela contribui para a saída?

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Com **h** cabeças paralelas, cada uma com suas próprias matrizes $W_Q^i, W_K^i, W_V^i$.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model deve ser divisível por num_heads"

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads  # dimensão por cabeça

        # Projeções lineares (uma para todo o conjunto de cabeças)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None  # para visualização

    def forward(self, x, mask=None):
        """
        x:    (batch, seq_len, d_model)
        mask: (batch, seq_len) — True nas posições de padding
        """
        B, N, _ = x.shape
        h, dk   = self.num_heads, self.d_k

        # 1. Projetar e reparticionar em h cabeças
        def project_split(linear, t):
            # (B, N, d_model) → (B, h, N, dk)
            return linear(t).view(B, N, h, dk).transpose(1, 2)

        Q = project_split(self.W_q, x)   # (B, h, N, dk)
        K = project_split(self.W_k, x)
        V = project_split(self.W_v, x)

        # 2. Scaled dot-product attention
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(dk)  # (B, h, N, N)

        if mask is not None:
            # mask: (B, N) → (B, 1, 1, N) para broadcast
            scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(2), float('-inf'))

        attn = F.softmax(scores, dim=-1)  # (B, h, N, N)
        attn = self.dropout(attn)
        self.attn_weights = attn.detach()  # guardar para visualização

        # 3. Combinar com Values e concatenar cabeças
        out = (attn @ V)                          # (B, h, N, dk)
        out = out.transpose(1, 2).contiguous()     # (B, N, h, dk)
        out = out.view(B, N, self.d_model)         # (B, N, d_model) — concatenação

        # 4. Projeção final W_O
        return self.W_o(out)                       # (B, N, d_model)


# Teste rápido
mha = MultiHeadAttention(d_model=64, num_heads=4)
x_test = torch.randn(2, 10, 64)  # batch=2, seq_len=10, d=64
out_test = mha(x_test)
print(f"Entrada: {x_test.shape}  →  Saída MHA: {out_test.shape}")
print(f"Pesos de atenção por cabeça: {mha.attn_weights.shape}  (B, heads, N, N)")

## 4. Bloco Encoder Completo

Cada bloco do encoder combina:
1. **Multi-Head Self-Attention** + conexão residual + LayerNorm
2. **Feed-Forward Network** (Linear → ReLU → Linear) + conexão residual + LayerNorm

In [ ]:
class FeedForward(nn.Module):
    """Sublayer FFN: Linear(d, 4d) → ReLU → Linear(4d, d)"""
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Um único bloco do Transformer Encoder."""
    def __init__(self, d_model: int, num_heads: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff      = FeedForward(d_model, d_ff, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sublayer 1: MHA + residual + norm
        x = self.norm1(x + self.drop(self.attn(x, mask)))
        # Sublayer 2: FFN + residual + norm
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


class TransformerEncoder(nn.Module):
    """N blocos empilhados de Transformer Encoder."""
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int = None, max_len: int = 512,
                 dropout: float = 0.1, pad_idx: int = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)
        self.layers    = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.d_model = d_model

    def forward(self, tokens, mask=None):
        # tokens: (B, N) — índices de vocabulário
        x = self.embedding(tokens) * math.sqrt(self.d_model)  # escala padrão
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x  # (B, N, d_model)


# Teste
enc = TransformerEncoder(vocab_size=1000, d_model=64, num_heads=4, num_layers=2)
tokens_test = torch.randint(1, 1000, (2, 15))  # batch=2, seq=15
enc_out = enc(tokens_test)
print(f"Tokens: {tokens_test.shape}  →  Encoder out: {enc_out.shape}")

## 5. Aplicação: NER com WikiANN

**Named Entity Recognition (NER)** é análoga ao slot filling da aula: classificar cada token em categorias semânticas.  
Usamos o dataset **WikiANN** (inglês), com 3 entidades: PER, ORG, LOC (formato BIO).

Arquitetura:
```
Tokens → Embedding + PosEnc → [Encoder × N] → Dense → Softmax por token → Rótulo NER
```

In [ ]:
from datasets import load_dataset

raw = load_dataset('unimelb-nlp/wikiann', 'en')  # parquet nativo, sem script legado
print(raw)
print("\nExemplo:")
ex = raw['train'][0]
print("Tokens:", ex['tokens'])
print("NER tags:", ex['ner_tags'])
print("\nMapa de rótulos:", raw['train'].features['ner_tags'].feature.names)

In [ ]:
# Construir vocabulário
from collections import Counter

PAD_IDX = 0
UNK_IDX = 1
SPECIAL = ['<PAD>', '<UNK>']
MAX_LEN = 64

counter = Counter()
for ex in raw['train']:
    counter.update(t.lower() for t in ex['tokens'])

vocab = SPECIAL + [w for w, c in counter.most_common() if c >= 2]
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

label_names = raw['train'].features['ner_tags'].feature.names  # O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC
NUM_LABELS = len(label_names)
print(f"Vocabulário: {VOCAB_SIZE} tokens")
print(f"Rótulos ({NUM_LABELS}): {label_names}")

In [ ]:
class NERDataset(Dataset):
    def __init__(self, split, max_len=MAX_LEN):
        self.examples = []
        for ex in raw[split]:
            toks = [word2idx.get(t.lower(), UNK_IDX) for t in ex['tokens']]
            tags = ex['ner_tags']
            # Truncar / pad
            toks = toks[:max_len] + [PAD_IDX] * max(0, max_len - len(toks))
            tags = tags[:max_len] + [-100]   * max(0, max_len - len(tags))  # -100 = ignorar no loss
            self.examples.append((toks, tags))

    def __len__(self):  return len(self.examples)

    def __getitem__(self, idx):
        toks, tags = self.examples[idx]
        return (torch.tensor(toks, dtype=torch.long),
                torch.tensor(tags, dtype=torch.long))


train_ds = NERDataset('train')
val_ds   = NERDataset('validation')
test_ds  = NERDataset('test')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)
test_loader  = DataLoader(test_ds,  batch_size=64)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

In [ ]:
class TransformerNER(nn.Module):
    """Transformer Encoder + classificador por token."""
    def __init__(self, vocab_size, d_model, num_heads, num_layers, num_labels,
                 d_ff=None, max_len=MAX_LEN, dropout=0.1):
        super().__init__()
        self.encoder    = TransformerEncoder(
            vocab_size, d_model, num_heads, num_layers, d_ff, max_len, dropout
        )
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(self, tokens):
        pad_mask = (tokens == PAD_IDX)           # (B, N)
        x = self.encoder(tokens, pad_mask)        # (B, N, d_model)
        return self.classifier(x)                 # (B, N, num_labels)


model = TransformerNER(
    vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
    num_layers=2, num_labels=NUM_LABELS, dropout=0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis: {total_params:,}")

## 6. Treinamento

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for tokens, labels in loader:
        tokens, labels = tokens.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(tokens)                   # (B, N, num_labels)
        # CrossEntropy espera (B*N, C) e (B*N,); ignora -100
        loss = criterion(logits.view(-1, NUM_LABELS), labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for tokens, labels in loader:
        tokens, labels = tokens.to(device), labels.to(device)
        preds = model(tokens).argmax(-1)  # (B, N)
        mask  = labels != -100
        correct += (preds[mask] == labels[mask]).sum().item()
        total   += mask.sum().item()
    return correct / total if total > 0 else 0.0


criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4, steps_per_epoch=len(train_loader), epochs=5
)

history = {'train_loss': [], 'val_acc': []}

for epoch in range(1, 6):
    loss = train_epoch(model, train_loader, optimizer, criterion)
    scheduler.step()
    acc  = evaluate(model, val_loader)
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    print(f"Epoch {epoch}/5  loss={loss:.4f}  val_acc={acc:.4f}")

test_acc = evaluate(model, test_loader)
print(f"\nTest accuracy: {test_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], marker='o', color='#3b82f6')
axes[0].set(title='Loss de treinamento', xlabel='Época', ylabel='CrossEntropy')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_acc'], marker='o', color='#10b981')
axes[1].set(title='Acurácia de validação', xlabel='Época', ylabel='Acc', ylim=(0, 1))
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Visualização dos Pesos de Atenção

In [ ]:
@torch.no_grad()
def show_attention(model, sentence: str, layer: int = -1):
    """Visualiza os pesos de atenção de cada cabeça para uma frase."""
    tokens_str = sentence.lower().split()
    token_ids  = [word2idx.get(t, UNK_IDX) for t in tokens_str]
    pad_needed = MAX_LEN - len(token_ids)
    token_ids  = token_ids + [PAD_IDX] * pad_needed

    inp = torch.tensor([token_ids], dtype=torch.long, device=device)
    model.eval()
    _ = model(inp)

    # Pegar pesos da última camada do encoder
    attn = model.encoder.layers[layer].attn.attn_weights  # (1, h, N, N)
    N = len(tokens_str)
    attn = attn[0, :, :N, :N].cpu().numpy()  # (h, N, N)

    num_heads = attn.shape[0]
    fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))
    if num_heads == 1:
        axes = [axes]

    for h_idx, ax in enumerate(axes):
        sns.heatmap(attn[h_idx], annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=tokens_str, yticklabels=tokens_str, ax=ax,
                    cbar=False, square=True, annot_kws={'size': 8})
        ax.set_title(f'Cabeça {h_idx+1}', fontsize=10)
        ax.tick_params(axis='both', labelsize=8)

    plt.suptitle(f'Pesos de atenção — camada {layer} — "{sentence}"', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()


show_attention(model, 'MVP lives in New York and works for Google')

## 8. Comparação: Transformer vs. BiLSTM

Vamos treinar um BiLSTM com o mesmo número aproximado de parâmetros para comparação.

In [ ]:
class BiLSTMNER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_labels, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.drop       = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, tokens):
        x   = self.drop(self.embedding(tokens))
        out, _ = self.lstm(x)
        return self.classifier(self.drop(out))


lstm_model = BiLSTMNER(
    vocab_size=VOCAB_SIZE, embed_dim=128, hidden_dim=64,
    num_layers=2, num_labels=NUM_LABELS
).to(device)

print(f"Parâmetros Transformer: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Parâmetros BiLSTM:      {sum(p.numel() for p in lstm_model.parameters() if p.requires_grad):,}")

In [ ]:
opt_lstm = optim.AdamW(lstm_model.parameters(), lr=3e-4)
sched_lstm = optim.lr_scheduler.OneCycleLR(
    opt_lstm, max_lr=3e-4, steps_per_epoch=len(train_loader), epochs=5
)

hist_lstm = {'train_loss': [], 'val_acc': []}

for epoch in range(1, 6):
    lstm_model.train()
    total_loss = 0
    for tokens, labels in train_loader:
        tokens, labels = tokens.to(device), labels.to(device)
        opt_lstm.zero_grad()
        logits = lstm_model(tokens)
        loss = criterion(logits.view(-1, NUM_LABELS), labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        opt_lstm.step()
        sched_lstm.step()
        total_loss += loss.item()
    val_acc = evaluate(lstm_model, val_loader)
    hist_lstm['train_loss'].append(total_loss / len(train_loader))
    hist_lstm['val_acc'].append(val_acc)
    print(f"[BiLSTM] Epoch {epoch}/5  loss={hist_lstm['train_loss'][-1]:.4f}  val_acc={val_acc:.4f}")

lstm_test_acc = evaluate(lstm_model, test_loader)
print(f"\nBiLSTM Test accuracy: {lstm_test_acc:.4f}")
print(f"Transformer Test accuracy: {test_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

epochs = range(1, 6)
axes[0].plot(epochs, history['train_loss'],      marker='o', label='Transformer', color='#3b82f6')
axes[0].plot(epochs, hist_lstm['train_loss'],    marker='s', label='BiLSTM',      color='#ef4444')
axes[0].set(title='Loss de treinamento', xlabel='Época', ylabel='CrossEntropy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['val_acc'],         marker='o', label='Transformer', color='#3b82f6')
axes[1].plot(epochs, hist_lstm['val_acc'],       marker='s', label='BiLSTM',      color='#ef4444')
axes[1].axhline(test_acc,      color='#3b82f6', linestyle='--', alpha=0.5, label=f'Transformer test ({test_acc:.3f})')
axes[1].axhline(lstm_test_acc, color='#ef4444', linestyle='--', alpha=0.5, label=f'BiLSTM test ({lstm_test_acc:.3f})')
axes[1].set(title='Acurácia de validação', xlabel='Época', ylabel='Acc', ylim=(0.5, 1.0))
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.suptitle('Transformer vs. BiLSTM — NER (CoNLL-2003)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Resumo

| Componente | Papel | Código |
|---|---|---|
| **Self-Attention** | Calcula similaridade entre todos os pares de tokens | `Q @ K.T / sqrt(dk)` → softmax → `@ V` |
| **Positional Encoding** | Adiciona informação de posição | Sinusoides pré-computadas somadas ao embedding |
| **Multi-Head Attention** | h padrões de atenção em paralelo | `num_heads` projeções independentes Q/K/V |
| **Add & Norm** | Estabiliza o gradiente em redes profundas | `LayerNorm(x + sublayer(x))` |
| **FFN** | Transforma cada token independentemente | `Linear(d, 4d) → ReLU → Linear(4d, d)` |

**Vantagens sobre RNNs:**
- Cada token acessa todos os outros diretamente (sem propagação sequencial)
- Totalmente paralelizável no treinamento
- Sem vanishing gradient no caminho da atenção

**Próxima aula:** BERT — pré-treino com Masked Language Modeling e fine-tuning.